# KTO：用非成对客服反馈做偏好优化

**面试问题：只有 desirable/undesirable 单样本时，sequence log-ratio、KL 基线和 loss aversion 怎样实现？**

## 回答主线

用一个可读任务先建立朴素 baseline，再从基础算子实现核心机制，输出中间状态、指标对照和失败修正。断言只在最后保护最关键的不变量；受控小数据用于解释机制，不冒充真实基础模型质量。

## 真实案例

客服日志没有 chosen/rejected 成对答案，只有用户对单条回复点“有帮助”或“没解决”。案例使用六条可读回复及 policy/reference sequence log-prob，先展示把绝对 log-prob 当质量分的错误基线，再实现 prompt-local log-ratio、KL reference point 和 KTO 的 gain/loss 两类效用，展示类别失衡加权和 stale reference 的失败。

### 输入预览：六条非成对反馈

In [1]:
import math  # 导入稳定 sigmoid 和对数损失需要的数学函数。

feedback = [  # 构造具有真实文本、标签和新旧策略序列 log-prob 的非成对样本。
    {"prompt": "退款多久到账", "response": "通常三个工作日到账", "label": "desirable", "policy_logp": -1.2, "reference_logp": -1.8},  # 有帮助回复被新策略提高概率。
    {"prompt": "退款多久到账", "response": "马上到账，放心", "label": "undesirable", "policy_logp": -1.4, "reference_logp": -2.2},  # 错误承诺也被新策略提高，需负反馈纠正。
    {"prompt": "订单未发货", "response": "我先帮您查询仓库状态", "label": "desirable", "policy_logp": -1.0, "reference_logp": -1.5},  # 正确行动计划获得正反馈。
    {"prompt": "订单未发货", "response": "请耐心等待", "label": "undesirable", "policy_logp": -1.7, "reference_logp": -1.9},  # 空泛回复只有轻微概率提升。
    {"prompt": "发票抬头修改", "response": "开票前可以修改，请提供新抬头", "label": "desirable", "policy_logp": -1.3, "reference_logp": -1.7},  # 事实正确且提供下一步。
    {"prompt": "发票抬头修改", "response": "任何时候都能修改", "label": "undesirable", "policy_logp": -1.6, "reference_logp": -2.1},  # 过度承诺的负反馈样本。
]  # 完成三组 prompt 的单条反馈日志。
print("非成对反馈日志：")  # 输出可读训练数据。
for row in feedback:  # 逐条展示文本、标签和两个策略概率。
    print(f'{row["label"]:<11} Δlogp={row["policy_logp"] - row["reference_logp"]:+.2f} | {row["prompt"]} -> {row["response"]}')  # 显示 KTO 真正使用的是相对 reference 的变化。

非成对反馈日志：
desirable   Δlogp=+0.60 | 退款多久到账 -> 通常三个工作日到账
undesirable Δlogp=+0.80 | 退款多久到账 -> 马上到账，放心
desirable   Δlogp=+0.50 | 订单未发货 -> 我先帮您查询仓库状态
undesirable Δlogp=+0.20 | 订单未发货 -> 请耐心等待
desirable   Δlogp=+0.40 | 发票抬头修改 -> 开票前可以修改，请提供新抬头
undesirable Δlogp=+0.50 | 发票抬头修改 -> 任何时候都能修改


## Baseline 基线：把绝对 Policy Log-prob 当质量

In [2]:
absolute_ranking = sorted(feedback, key=lambda row: row["policy_logp"], reverse=True)  # 按新策略绝对序列概率排序构造错误基线。
print("绝对 policy log-prob 排名：")  # 输出错误质量代理。
for rank, row in enumerate(absolute_ranking, start=1):  # 逐条展示长度和 prompt 差异被混入的排名。
    print(f'{rank}. logp={row["policy_logp"]:+.2f} label={row["label"]:<11} {row["response"]}')  # 显示高概率并不保证回复正确。
baseline_false_positive = absolute_ranking.index(next(row for row in feedback if row["response"] == "马上到账，放心")) + 1  # 获取危险承诺在绝对概率榜的位置。
print(f"危险回复“马上到账”在绝对概率榜排名={baseline_false_positive}，说明绝对 log-prob 不是偏好奖励。")  # 明确 baseline 失败原因。

绝对 policy log-prob 排名：
1. logp=-1.00 label=desirable   我先帮您查询仓库状态
2. logp=-1.20 label=desirable   通常三个工作日到账
3. logp=-1.30 label=desirable   开票前可以修改，请提供新抬头
4. logp=-1.40 label=undesirable 马上到账，放心
5. logp=-1.60 label=undesirable 任何时候都能修改
6. logp=-1.70 label=undesirable 请耐心等待
危险回复“马上到账”在绝对概率榜排名=4，说明绝对 log-prob 不是偏好奖励。


### 核心实现：Log-ratio、KL Reference Point 与 KTO 效用

In [3]:
def sigmoid(value):  # 实现数值稳定的标量 sigmoid。
    if value >= 0:  # 正数分支避免 exp 正向溢出。
        return 1.0 / (1.0 + math.exp(-value))  # 返回正数稳定形式。
    exponential = math.exp(value)  # 负数分支只计算小指数。
    return exponential / (1.0 + exponential)  # 返回负数稳定形式。

log_ratios = [row["policy_logp"] - row["reference_logp"] for row in feedback]  # 计算新策略相对 reference 的 sequence log-ratio。
kl_reference = sum(max(ratio, 0.0) for ratio in log_ratios) / len(log_ratios)  # 用批内正 log-ratio 构造非负 KL reference point 教学估计。

def kto_utility(row, reference_point, beta=1.5, loss_aversion=1.8):  # 实现 desirable gain 与 undesirable loss 的 KTO 风格效用。
    ratio = row["policy_logp"] - row["reference_logp"]  # 计算当前样本相对 reference 的概率变化。
    centered = ratio - reference_point  # 用 KL reference point 中心化策略变化。
    if row["label"] == "desirable":  # 正反馈希望策略概率高于 reference point。
        return sigmoid(beta * centered)  # 返回 desirable 的 gain utility。
    return loss_aversion * sigmoid(-beta * centered)  # 对负反馈用更高 loss-aversion 惩罚错误概率提升。

utilities = [kto_utility(row, kl_reference) for row in feedback]  # 对六条非成对反馈计算 KTO 效用。
print(f"批内 KL reference point={kl_reference:.3f}")  # 展示效用中心不是固定零点。
print("KTO 样本效用：")  # 输出逐样本 gain/loss。
for row, ratio, utility in zip(feedback, log_ratios, utilities):  # 对齐展示标签、log-ratio 和效用。
    print(f'{row["label"]:<11} ratio={ratio:+.2f} utility={utility:.3f} | {row["response"]}')  # 显示错误回复被提升时受到更强惩罚。

批内 KL reference point=0.500
KTO 样本效用：
desirable   ratio=+0.60 utility=0.537 | 通常三个工作日到账
undesirable ratio=+0.80 utility=0.701 | 马上到账，放心
desirable   ratio=+0.50 utility=0.500 | 我先帮您查询仓库状态
undesirable ratio=+0.20 utility=1.099 | 请耐心等待
desirable   ratio=+0.40 utility=0.463 | 开票前可以修改，请提供新抬头
undesirable ratio=+0.50 utility=0.900 | 任何时候都能修改


## 结果解读：类别失衡与梯度方向

In [4]:
positive_count = sum(row["label"] == "desirable" for row in feedback)  # 统计正反馈样本数。
negative_count = len(feedback) - positive_count  # 统计负反馈样本数。
class_weights = {"desirable": len(feedback) / (2 * positive_count), "undesirable": len(feedback) / (2 * negative_count)}  # 构造使两类总权重相等的采样权重。
weighted_utilities = [utility * class_weights[row["label"]] for row, utility in zip(feedback, utilities)]  # 对每条效用应用类别平衡权重。
desired_direction = ["提高概率" if row["label"] == "desirable" else "降低概率" for row in feedback]  # 把优化方向转成可读文本。
print("标签          类别权重  当前ratio  优化方向  加权效用")  # 输出逐样本训练解释表。
for row, ratio, direction, utility in zip(feedback, log_ratios, desired_direction, weighted_utilities):  # 逐条展示 KTO 如何处理非成对样本。
    print(f'{row["label"]:<12} {class_weights[row["label"]]:8.3f} {ratio:+9.2f} {direction:<8} {utility:8.3f}')  # 展示类别平衡和 loss aversion。
print("解读：KTO 不需要同 prompt 成对回答，但仍需要冻结 reference、可靠标签、prompt/group 切分和序列 mask。")  # 给出数据合同边界。

标签          类别权重  当前ratio  优化方向  加权效用
desirable       1.000     +0.60 提高概率        0.537
undesirable     1.000     +0.80 降低概率        0.701
desirable       1.000     +0.50 提高概率        0.500
undesirable     1.000     +0.20 降低概率        1.099
desirable       1.000     +0.40 提高概率        0.463
undesirable     1.000     +0.50 降低概率        0.900
解读：KTO 不需要同 prompt 成对回答，但仍需要冻结 reference、可靠标签、prompt/group 切分和序列 mask。


## 失败案例：混用旧 Reference 版本改变奖励坐标

In [5]:
stale_reference = [-2.8, -3.2, -2.5, -2.7, -2.6, -3.1]  # 构造来自更旧且整体低概率的 reference log-prob。
stale_ratios = [row["policy_logp"] - old_logp for row, old_logp in zip(feedback, stale_reference)]  # 计算错误 reference 下虚高的 log-ratio。
stale_kl = sum(max(ratio, 0.0) for ratio in stale_ratios) / len(stale_ratios)  # 计算错误坐标系的 KL reference point。
danger_index = next(index for index, row in enumerate(feedback) if row["response"] == "马上到账，放心")  # 定位危险负反馈样本。
correct_danger_utility = utilities[danger_index]  # 读取正确 reference 下的负反馈效用。
stale_row = dict(feedback[danger_index], reference_logp=stale_reference[danger_index])  # 构造混用旧 reference 的同一训练样本。
stale_danger_utility = kto_utility(stale_row, stale_kl)  # 计算错误 reference 下的效用。
print(f"正确 KL={kl_reference:.3f}，旧 reference KL={stale_kl:.3f}")  # 展示坐标漂移幅度。
print(f"危险回复效用：正确={correct_danger_utility:.3f}，混用旧reference={stale_danger_utility:.3f}")  # 展示 reward 语义发生变化。
print("修正：训练 batch 的 policy/ref log-prob、tokenizer、template 和 checkpoint revision 必须作为同一不可分割制品。")  # 给出版本门禁。

正确 KL=0.500，旧 reference KL=1.450
危险回复效用：正确=0.701，混用旧reference=0.669
修正：训练 batch 的 policy/ref log-prob、tokenizer、template 和 checkpoint revision 必须作为同一不可分割制品。


### 生产边界

In [6]:
training_manifest = {"objective": "KTO", "beta": 1.5, "loss_aversion": 1.8, "policy_revision": "support-r7", "reference_revision": "support-r6", "template": "chat-v4", "labels": {"desirable": positive_count, "undesirable": negative_count}, "kl_reference": round(kl_reference, 4)}  # 构造可复现 KTO 训练清单。
print("KTO manifest：", training_manifest)  # 展示 objective 超参数、版本和标签分布。
print("生产替换点：真实训练需 token-level mask、分布式稳定聚合、holdout prompt、人工标签审计、KL 监控和独立安全评测。")  # 明确标量样本与大模型优化的差距。

KTO manifest： {'objective': 'KTO', 'beta': 1.5, 'loss_aversion': 1.8, 'policy_revision': 'support-r7', 'reference_revision': 'support-r6', 'template': 'chat-v4', 'labels': {'desirable': 3, 'undesirable': 3}, 'kl_reference': 0.5}
生产替换点：真实训练需 token-level mask、分布式稳定聚合、holdout prompt、人工标签审计、KL 监控和独立安全评测。


## 回归测试：只保护坐标、标签方向和版本探针

In [7]:
assert len(log_ratios) == len(feedback)  # 验证每条反馈都有 policy/reference 配对分数。
assert kl_reference >= 0.0  # 验证教学 KL reference point 保持非负。
assert all(direction == ("提高概率" if row["label"] == "desirable" else "降低概率") for row, direction in zip(feedback, desired_direction))  # 验证两类标签的优化方向没有反转。
assert abs(sum(class_weights[row["label"]] for row in feedback if row["label"] == "desirable") - sum(class_weights[row["label"]] for row in feedback if row["label"] == "undesirable")) < 1e-12  # 验证类别总权重平衡。
assert abs(stale_danger_utility - correct_danger_utility) > 1e-3  # 验证 reference 混用失败探针确实改变效用。
print("回归测试通过：log-ratio、KL、标签方向、类别平衡和 reference 版本探针均成立。")  # 用少量断言总结 KTO 数据合同。

回归测试通过：log-ratio、KL、标签方向、类别平衡和 reference 版本探针均成立。
